# 06 — Sanity Check: Leakage, Score Plausibility, SHAP Plausibility
**ZivaBasa MVP (Kaggle-Data Phase)**

Before trusting any result from notebooks 03/04/05, run this. It automates the three checks that
actually matter on proxy data:

1. **Target distribution** — is each task's target sane (not degenerate, not all-one-class)?
2. **Leakage scan** — does any feature correlate suspiciously highly with its task's target?
3. **Score plausibility** — are baseline/NN scores in a believable range, or suspiciously perfect?
4. **SHAP plausibility** — do the top SHAP features make domain sense, and do they roughly agree
   with the tree-based feature importances from notebook 03?

This notebook doesn't fix anything — it flags. A flag means "look at this before you trust it,"
not automatically "something is wrong." On proxy Kaggle data, some flags are expected and fine
once you understand why (e.g. weak signal genuinely produces near-random scores).

**Input:** everything notebooks 02–05 already produced. Run this after 05, not standalone.


In [1]:
# --- Setup ---
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from scipy.stats import spearmanr

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../models"
SHAP_DIR = "../models/shap_outputs"

TASK_CONFIG = {
    "employment": {
        "target": "target_high_automation_risk",
        "task_type": "classification",
        "drop_cols": ["target_high_automation_risk", "automation_risk_score",
                      "automation_exposure_index", "exposure_x_skill_complexity"],
    },
    "skills": {
        "target": "target_attrition",
        "task_type": "classification",
        "drop_cols": ["target_attrition"],
    },
    "productivity": {
        "target": "target_ai_adoption",
        "task_type": "regression",
        "drop_cols": ["target_ai_adoption", "ai_adoption_level", "ai_adoption_index"],
    },
}

# Collects every flag raised across all sections, printed as one report at the end
flags = []  # list of dicts: {"section": str, "task": str, "severity": "warn"|"fail", "message": str}

def flag(section, task, severity, message):
    flags.append({"section": section, "task": task, "severity": severity, "message": message})
    icon = "🔴" if severity == "fail" else "🟡"
    print(f"{icon} [{section}/{task}] {message}")

def ok(section, task, message):
    print(f"🟢 [{section}/{task}] {message}")


## 1. Target Distribution Check

A degenerate target (e.g. 99% one class, or zero variance) makes every downstream metric
meaningless regardless of model quality. Check this first — it's the cheapest check and the one
most likely to explain a confusing result later.


In [2]:
def load_features(name):
    path = os.path.join(PROCESSED_DIR, f"{name}_features.parquet")
    if not os.path.exists(path):
        print(f"[MISSING] {path} — run notebook 02 first.")
        return None
    return pd.read_parquet(path)

processed = {name: load_features(name) for name in TASK_CONFIG}

for name, cfg in TASK_CONFIG.items():
    df = processed[name]
    if df is None or cfg["target"] not in df.columns:
        flag("target_distribution", name, "fail", "Target column missing — nothing downstream can be trusted until this is fixed.")
        continue

    y = df[cfg["target"]]
    if cfg["task_type"] == "classification":
        counts = y.value_counts(normalize=True)
        minority_pct = counts.min() * 100
        print(f"[{name}] class balance:\n{(counts * 100).round(1)}")
        if minority_pct < 2:
            flag("target_distribution", name, "fail", f"Minority class is only {minority_pct:.1f}% — severe imbalance, metrics like accuracy will be misleading.")
        elif minority_pct < 10:
            flag("target_distribution", name, "warn", f"Minority class is {minority_pct:.1f}% — imbalanced; prefer ROC-AUC/F1 over accuracy, consider class_weight='balanced'.")
        else:
            ok("target_distribution", name, f"Class balance looks reasonable (minority class {minority_pct:.1f}%).")
    else:
        desc = y.describe()
        print(f"[{name}] target summary:\n{desc}")
        if desc["std"] < 1e-6:
            flag("target_distribution", name, "fail", "Target has ~zero variance — regression metrics are meaningless.")
        else:
            ok("target_distribution", name, f"Target has real variance (std={desc['std']:.4f}).")


[employment] class balance:
target_high_automation_risk
0    75.0
1    25.0
Name: proportion, dtype: float64
🟢 [target_distribution/employment] Class balance looks reasonable (minority class 25.0%).
[skills] class balance:
target_attrition
False    83.9
True     16.1
Name: proportion, dtype: float64
🟢 [target_distribution/skills] Class balance looks reasonable (minority class 16.1%).
[productivity] target summary:
count    1.500000e+04
mean     9.947598e-18
std      1.000033e+00
min     -1.693555e+00
25%     -8.698868e-01
50%     -1.258912e-02
75%      8.666372e-01
max      1.706167e+00
Name: target_ai_adoption, dtype: float64
🟢 [target_distribution/productivity] Target has real variance (std=1.0000).


## 2. Leakage Scan — Feature/Target Correlation

Flags any feature whose raw correlation with the target exceeds a threshold. This is the same
check that caught the `exposure_x_skill_complexity` leakage bug during development — it's cheap,
automatic, and catches the most common failure mode on engineered features.

Threshold is deliberately aggressive (0.3) since even "reasonable" engineered features on proxy
data shouldn't usually correlate that strongly with a threshold-derived or otherwise noisy target.
Adjust `CORR_THRESHOLD` if you have a specific reason to expect a strong legitimate driver.


In [3]:
CORR_THRESHOLD = 0.30

for name, cfg in TASK_CONFIG.items():
    df = processed[name]
    if df is None or cfg["target"] not in df.columns:
        continue

    drop_cols = [c for c in cfg["drop_cols"] if c in df.columns]
    X = df.drop(columns=drop_cols).select_dtypes(include=[np.number])
    y = df[cfg["target"]]

    corrs = X.corrwith(y).abs().sort_values(ascending=False)
    suspicious = corrs[corrs > CORR_THRESHOLD]

    print(f"\n[{name}] top 5 feature-target correlations:")
    print(corrs.head(5).round(3))

    if len(suspicious) > 0:
        for feat, corr_val in suspicious.items():
            flag("leakage_scan", name, "warn",
                 f"'{feat}' correlates {corr_val:.3f} with target — verify this isn't derived from the target's raw source column.")
    else:
        ok("leakage_scan", name, f"No feature exceeds |corr| > {CORR_THRESHOLD} with target.")



[employment] top 5 feature-target correlations:
percent_tasks_automatable    0.041
training_hours_needed        0.028
skill_complexity_score       0.026
job_demand_index             0.016
ai_tool_maturity_score       0.014
dtype: float64
🟢 [leakage_scan/employment] No feature exceeds |corr| > 0.3 with target.

[skills] top 5 feature-target correlations:
MonthlyIncome               0.160
Age                         0.159
training_intensity_index    0.158
YearsAtCompany              0.139
JobSatisfaction             0.103
dtype: float64
🟢 [leakage_scan/skills] No feature exceeds |corr| > 0.3 with target.

[productivity] top 5 feature-target correlations:
skill_gap_index    0.003
dtype: float64
🟢 [leakage_scan/productivity] No feature exceeds |corr| > 0.3 with target.


## 3. Score Plausibility — Baselines and Multi-Task NN

Flags scores that are **suspiciously high** (a leakage signal on proxy data — Section 9.2 of the
project documentation found a real bug exactly this way) as well as scores that are **exactly
chance level across every single model**, which usually means the target has no learnable signal
at all rather than a healthy "hard problem" result.


In [4]:
SUSPICIOUS_HIGH = {"roc_auc": 0.93, "accuracy": 0.93, "r2": 0.85}
CHANCE_LEVEL = {"roc_auc": (0.45, 0.55)}

def load_csv_safe(path):
    if not os.path.exists(path):
        print(f"[MISSING] {path}")
        return None
    return pd.read_csv(path)

baseline_results = load_csv_safe(os.path.join(MODELS_DIR, "baseline_results.csv"))
nn_results = load_csv_safe(os.path.join(MODELS_DIR, "multitask_model", "multitask_nn_results.csv"))

all_results = pd.concat([r for r in [baseline_results, nn_results] if r is not None], ignore_index=True) \
    if (baseline_results is not None or nn_results is not None) else None

if all_results is not None:
    display(all_results)

    for task_name in TASK_CONFIG:
        task_rows = all_results[all_results["task_head"] == task_name]
        if task_rows.empty:
            continue

        task_type = TASK_CONFIG[task_name]["task_type"]
        metric_col = "roc_auc" if task_type == "classification" else "r2"
        if metric_col not in task_rows.columns:
            continue
        vals = task_rows[metric_col].dropna()
        if vals.empty:
            continue

        # Suspiciously high (possible leakage)
        threshold = SUSPICIOUS_HIGH.get(metric_col)
        if threshold and vals.max() > threshold:
            best_model = task_rows.loc[task_rows[metric_col].idxmax(), "model"]
            flag("score_plausibility", task_name, "warn",
                 f"{best_model} scores {metric_col}={vals.max():.3f} — unusually high for proxy data. "
                 f"Re-check Section 2 (leakage scan) and the feature dictionary before trusting this.")

        # All models stuck at chance level (classification only)
        if metric_col == "roc_auc":
            lo, hi = CHANCE_LEVEL["roc_auc"]
            if (vals > lo).sum() == 0 or (vals < hi).all():
                if vals.between(lo, hi).all():
                    flag("score_plausibility", task_name, "warn",
                         f"Every model is within chance-level ROC-AUC ({lo}-{hi}) — target may have no learnable "
                         f"signal from these features, or the target definition itself may need revisiting.")

        if not (threshold and vals.max() > threshold) and not (metric_col == "roc_auc" and vals.between(*CHANCE_LEVEL["roc_auc"]).all()):
            ok("score_plausibility", task_name, f"Score range for {metric_col} looks like a believable, non-trivial result.")
else:
    flag("score_plausibility", "all", "fail", "No baseline or NN results found — run notebooks 03 and 04 first.")


,task_head,model,accuracy,precision,recall,f1,roc_auc,rmse,mae,r2
0,employment,logistic_regression,0.750000,0.000000,0.000000,0.000000,0.478163,NaN,NaN,NaN
1,employment,decision_tree,0.710000,0.250000,0.080000,0.121212,0.495163,NaN,NaN,NaN
2,employment,random_forest,0.750000,0.500000,0.006667,0.013158,0.457867,NaN,NaN,NaN
3,employment,gradient_boosting,0.731667,0.210526,0.026667,0.047337,0.488504,NaN,NaN,NaN
4,skills,logistic_regression,0.826531,0.250000,0.042553,0.072727,0.680076,NaN,NaN,NaN
5,skills,decision_tree,0.772109,0.166667,0.106383,0.129870,0.500775,NaN,NaN,NaN
6,skills,random_forest,0.823129,0.368421,0.148936,0.212121,0.646524,NaN,NaN,NaN
7,skills,gradient_boosting,0.795918,0.303030,0.212766,0.250000,0.598717,NaN,NaN,NaN
8,productivity,linear_regression,NaN,NaN,NaN,NaN,NaN,0.993797,0.863194,-0.001151
9,productivity,decision_tree,NaN,NaN,NaN,NaN,NaN,1.006364,0.869790,-0.026630


🟢 [score_plausibility/employment] Score range for roc_auc looks like a believable, non-trivial result.
🟢 [score_plausibility/skills] Score range for roc_auc looks like a believable, non-trivial result.
🟢 [score_plausibility/productivity] Score range for r2 looks like a believable, non-trivial result.


## 4. Baseline vs. Multi-Task NN — Does the Deep Model Earn Its Complexity?

Not a pass/fail flag, just the comparison itself — worth looking at directly rather than only
through the automated checks above.


In [5]:
if all_results is not None:
    for task_name in TASK_CONFIG:
        task_rows = all_results[all_results["task_head"] == task_name]
        if task_rows.empty:
            continue
        task_type = TASK_CONFIG[task_name]["task_type"]
        metric_col = "roc_auc" if task_type == "classification" else "r2"
        if metric_col not in task_rows.columns or task_rows[metric_col].dropna().empty:
            continue

        sorted_rows = task_rows.sort_values(metric_col, ascending=False)
        best = sorted_rows.iloc[0]
        nn_row = task_rows[task_rows["model"] == "multitask_nn"]

        print(f"=== {task_name} ({metric_col}) ===")
        print(sorted_rows[["model", metric_col]].to_string(index=False))
        if not nn_row.empty and best["model"] != "multitask_nn":
            gap = best[metric_col] - nn_row.iloc[0][metric_col]
            print(f"  -> Best baseline beats multi-task NN by {gap:.3f} on {metric_col}. "
                  f"Worth noting explicitly in any write-up rather than only reporting the NN's number.")
        print()


=== employment (roc_auc) ===
              model  roc_auc
      decision_tree 0.495163
  gradient_boosting 0.488504
logistic_regression 0.478163
      random_forest 0.457867
       multitask_nn 0.442370
  -> Best baseline beats multi-task NN by 0.053 on roc_auc. Worth noting explicitly in any write-up rather than only reporting the NN's number.

=== skills (roc_auc) ===
              model  roc_auc
logistic_regression 0.680076
      random_forest 0.646524
       multitask_nn 0.605909
  gradient_boosting 0.598717
      decision_tree 0.500775
  -> Best baseline beats multi-task NN by 0.074 on roc_auc. Worth noting explicitly in any write-up rather than only reporting the NN's number.

=== productivity (r2) ===
            model        r2
     multitask_nn -0.000528
linear_regression -0.001151
    random_forest -0.008041
gradient_boosting -0.010970
    decision_tree -0.026630



## 5. SHAP Plausibility

Two checks: (a) do the top SHAP features look like things a domain expert would expect to matter,
and (b) do they roughly agree with the tree-based feature importances from notebook 03 — if SHAP
and Random Forest strongly disagree on what matters, that's worth understanding before either is
trusted for an HR-facing explanation.


In [6]:
def load_shap_importance(name):
    path = os.path.join(SHAP_DIR, f"{name}_feature_importance.csv")
    if not os.path.exists(path):
        print(f"[MISSING] {path} — run notebook 05 first.")
        return None
    return pd.read_csv(path)

for name in TASK_CONFIG:
    shap_imp = load_shap_importance(name)
    if shap_imp is None:
        continue
    print(f"=== {name} — Top 10 SHAP Features ===")
    display(shap_imp.head(10))
    ok("shap_plausibility", name, "Top SHAP features printed above — review for domain sense manually; this can't be fully automated.")


=== employment — Top 10 SHAP Features ===


,feature,mean_abs_shap
0,skill_complexity_score,0.020197
1,training_hours_needed,0.019079
2,job_demand_index,0.019016
3,percent_tasks_automatable,0.017666
4,task_repetition_level,0.013053
5,avg_salary_usd,0.012510
6,ai_tool_maturity_score,0.010611


🟢 [shap_plausibility/employment] Top SHAP features printed above — review for domain sense manually; this can't be fully automated.
=== skills — Top 10 SHAP Features ===


,feature,mean_abs_shap
0,Age,0.051019
1,training_intensity_index,0.047678
2,TrainingTimesLastYear,0.042264
3,MonthlyIncome,0.032545
4,JobSatisfaction,0.031402
5,YearsAtCompany,0.020888
6,training_x_satisfaction,0.012406
7,PerformanceRating,0.007053


🟢 [shap_plausibility/skills] Top SHAP features printed above — review for domain sense manually; this can't be fully automated.
=== productivity — Top 10 SHAP Features ===


,feature,mean_abs_shap
0,skill_gap_index,0.048357


🟢 [shap_plausibility/productivity] Top SHAP features printed above — review for domain sense manually; this can't be fully automated.


In [7]:
# Rank correlation between SHAP importance and Random Forest feature_importances_,
# where both are available. Low/negative correlation is a flag worth investigating, not
# necessarily a bug -- SHAP and impurity-based importance can legitimately diverge.
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split

for name, cfg in TASK_CONFIG.items():
    shap_imp = load_shap_importance(name)
    df = processed[name]
    if shap_imp is None or df is None or cfg["target"] not in df.columns:
        continue

    drop_cols = [c for c in cfg["drop_cols"] if c in df.columns]
    X = df.drop(columns=drop_cols).select_dtypes(include=[np.number])
    y = df[cfg["target"]]

    rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42) \
        if cfg["task_type"] == "classification" else \
        RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
    rf.fit(X, y)
    rf_imp = pd.Series(rf.feature_importances_, index=X.columns, name="rf_importance")

    merged = shap_imp.set_index("feature")[["mean_abs_shap"]].join(rf_imp, how="inner")
    if len(merged) < 3:
        print(f"[{name}] too few overlapping features to compute rank correlation.")
        continue

    rho, pval = spearmanr(merged["mean_abs_shap"], merged["rf_importance"])
    print(f"[{name}] SHAP vs. Random Forest importance rank correlation: rho={rho:.3f} (p={pval:.3f})")
    if rho < 0.3:
        flag("shap_plausibility", name, "warn",
             f"SHAP and Random Forest importances only weakly agree (rho={rho:.3f}). "
             f"Not necessarily wrong, but worth understanding why before using SHAP output in an HR-facing explanation.")
    else:
        ok("shap_plausibility", name, f"SHAP and Random Forest importances broadly agree (rho={rho:.3f}).")


[employment] SHAP vs. Random Forest importance rank correlation: rho=-0.607 (p=0.148)
🟡 [shap_plausibility/employment] SHAP and Random Forest importances only weakly agree (rho=-0.607). Not necessarily wrong, but worth understanding why before using SHAP output in an HR-facing explanation.
[skills] SHAP vs. Random Forest importance rank correlation: rho=0.333 (p=0.420)
🟢 [shap_plausibility/skills] SHAP and Random Forest importances broadly agree (rho=0.333).
[productivity] too few overlapping features to compute rank correlation.


## 5b. Diagnosing SHAP vs. Random Forest Disagreement

Runs automatically for any task flagged in Section 5. A weak or negative rank correlation has at
least three different possible causes, and they call for different responses:

1. **Genuine multi-task entanglement** — the neural network's employment path shares its trunk
   with skills and productivity, so its learned representation can legitimately differ from a
   standalone Random Forest trained only on employment data. Not a bug, but worth stating
   explicitly wherever these SHAP values are used.
2. **KernelExplainer approximation noise** — the fallback explainer (used because GradientExplainer
   is incompatible with this TF/Keras version) approximates Shapley values from a limited sample
   budget (`nsamples=100`). With few features, small sampling noise can flip ranks easily.
3. **Small-N instability in the rank correlation itself** — Spearman's rho on a handful of
   features is inherently noisy; a "disagreement" can be within the range you'd see from chance
   alone.

This section separates cause #2/#3 (noise) from cause #1 (real) by recomputing SHAP with a much
larger sample budget and checking whether rho actually moves.


In [8]:
SHAP_DISAGREEMENT_THRESHOLD = 0.30
STABILITY_CHECK_NSAMPLES = 500  # vs. 100 in the main notebook 05 run

for name, cfg in TASK_CONFIG.items():
    shap_imp = load_shap_importance(name)
    df = processed[name]
    if shap_imp is None or df is None or cfg["target"] not in df.columns:
        continue

    drop_cols = [c for c in cfg["drop_cols"] if c in df.columns]
    X = df.drop(columns=drop_cols).select_dtypes(include=[np.number])
    y = df[cfg["target"]]

    rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42) \
        if cfg["task_type"] == "classification" else \
        RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
    rf.fit(X, y)
    rf_imp = pd.Series(rf.feature_importances_, index=X.columns, name="rf_importance")

    merged = shap_imp.set_index("feature")[["mean_abs_shap"]].join(rf_imp, how="inner")
    if len(merged) < 3:
        continue
    rho, pval = spearmanr(merged["mean_abs_shap"], merged["rf_importance"])

    if abs(rho) >= SHAP_DISAGREEMENT_THRESHOLD:
        continue  # not flagged, nothing to diagnose

    print(f"=== Diagnosing {name}: rho={rho:.3f}, p={pval:.3f}, n_features={len(merged)} ===\n")

    # --- Step 1: side-by-side ranking table ---
    merged_ranked = merged.copy()
    merged_ranked["shap_rank"] = merged_ranked["mean_abs_shap"].rank(ascending=False).astype(int)
    merged_ranked["rf_rank"] = merged_ranked["rf_importance"].rank(ascending=False).astype(int)
    merged_ranked["rank_gap"] = (merged_ranked["shap_rank"] - merged_ranked["rf_rank"]).abs()
    print("Side-by-side ranking (sorted by SHAP rank):")
    display(merged_ranked.sort_values("shap_rank")[["shap_rank", "rf_rank", "rank_gap", "mean_abs_shap", "rf_importance"]])

    # --- Step 2: statistical significance context ---
    if len(merged) < 15:
        print(f"NOTE: only {len(merged)} overlapping features -- Spearman rho is inherently unstable "
              f"at this N. A p-value of {pval:.3f} means this correlation {'is NOT' if pval > 0.05 else 'IS'} "
              f"statistically distinguishable from zero at the usual 0.05 threshold. Treat the sign/magnitude "
              f"with real caution either way.\n")

    # --- Step 3: stability check -- does rho change with a much larger SHAP sample budget? ---
    try:
        keras_model_path = os.path.join(MODELS_DIR, "multitask_model", f"{name}_model.keras")
        if os.path.exists(keras_model_path):
            import tensorflow as tf
            keras_model = tf.keras.models.load_model(keras_model_path)

            rng = np.random.RandomState(42)
            X_arr = X.values.astype("float32")
            bg_idx = rng.choice(len(X_arr), size=min(100, len(X_arr)), replace=False)
            ex_idx = rng.choice(len(X_arr), size=min(50, len(X_arr)), replace=False)
            background = X_arr[bg_idx]
            explain_set = X_arr[ex_idx]

            predict_fn = lambda x: keras_model.predict(x, verbose=0).reshape(-1)
            bg_summary = shap.kmeans(background, min(20, background.shape[0]))
            explainer = shap.KernelExplainer(predict_fn, bg_summary)
            sv_stable = explainer.shap_values(explain_set, nsamples=STABILITY_CHECK_NSAMPLES)
            sv_stable = np.squeeze(np.array(sv_stable))

            stable_imp = pd.Series(np.abs(sv_stable).mean(axis=0), index=X.columns, name="mean_abs_shap_stable")
            merged_stable = pd.DataFrame(stable_imp).join(rf_imp, how="inner")
            rho_stable, pval_stable = spearmanr(merged_stable["mean_abs_shap_stable"], merged_stable["rf_importance"])

            print(f"Stability check: rho at nsamples=100 (original) = {rho:.3f}")
            print(f"                 rho at nsamples={STABILITY_CHECK_NSAMPLES} (stability check) = {rho_stable:.3f}")
            if abs(rho_stable - rho) > 0.25:
                flag("shap_plausibility_diagnosis", name, "warn",
                     f"rho shifted substantially ({rho:.3f} -> {rho_stable:.3f}) with a larger SHAP sample budget -- "
                     f"the original disagreement looks like KernelExplainer sampling noise, not a real signal. "
                     f"Consider re-running notebook 05 with a higher nsamples for this task.")
            else:
                flag("shap_plausibility_diagnosis", name, "warn",
                     f"rho stayed similar ({rho:.3f} -> {rho_stable:.3f}) even with more SHAP samples -- "
                     f"this looks like a REAL disagreement between the multi-task NN and a standalone Random Forest, "
                     f"most likely explained by shared-trunk entanglement with the other tasks (see cause #1 above), "
                     f"not sampling noise.")
        else:
            print(f"[{name}] saved model not found at {keras_model_path}, skipping stability check.")
    except Exception as e:
        print(f"[{name}] stability check failed ({type(e).__name__}: {e}) -- inconclusive, not treated as a flag.")
    print()


## 6. Summary Report


In [9]:
print("=" * 70)
print("SANITY CHECK SUMMARY")
print("=" * 70)

if not flags:
    print("\n🟢 No flags raised. All checks passed cleanly.")
else:
    fails = [f for f in flags if f["severity"] == "fail"]
    warns = [f for f in flags if f["severity"] == "warn"]
    print(f"\n{len(fails)} FAIL, {len(warns)} WARN\n")
    if fails:
        print("🔴 FAIL (fix before trusting results):")
        for f in fails:
            print(f"   [{f['section']}/{f['task']}] {f['message']}")
    if warns:
        print("\n🟡 WARN (review, may be fine):")
        for f in warns:
            print(f"   [{f['section']}/{f['task']}] {f['message']}")

print("""

Remember throughout: this is proxy Kaggle data standing in for real banking-sector data.
A "clean" result here validates the PIPELINE, not real workforce findings. Keep that
distinction explicit in anything downstream of this notebook.
""")


SANITY CHECK SUMMARY

0 FAIL, 1 WARN


🟡 WARN (review, may be fine):
   [shap_plausibility/employment] SHAP and Random Forest importances only weakly agree (rho=-0.607). Not necessarily wrong, but worth understanding why before using SHAP output in an HR-facing explanation.


Remember throughout: this is proxy Kaggle data standing in for real banking-sector data.
A "clean" result here validates the PIPELINE, not real workforce findings. Keep that
distinction explicit in anything downstream of this notebook.

